# LeetCode 4088 — Difference Between Peaks and Valleys bằng Python

Link đề: https://leetcode.com/problems/difference-between-peaks-and-valleys/ · Độ khó: **Easy** · Chủ đề: Array, Simulation (duyệt mảng, so sánh hàng xóm)

Skeleton LeetCode:

```python
class Solution:
    def peakValleyDifference(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc (Biweekly Contest 194, Q1); mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force viết độc lập theo đúng định nghĩa.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một vị trí `i` là **đỉnh (peak)** nếu nó **lớn hơn cả hai hàng xóm**, là **đáy (valley)** nếu **nhỏ hơn cả hai hàng xóm**; hai đầu mảng **không bao giờ** được tính.
2. Biết đọc/viết điều kiện "so với hàng xóm" bằng **so sánh dây chuyền** `a < b > c` và biết khi nào dấu `<`, `>` phải là **nghiêm ngặt** (bằng nhau thì không tính).
3. Đi từ cách **tách hai danh sách đỉnh/đáy rồi trừ** → **một lượt duyệt cộng/trừ trực tiếp** → **`zip` ba lát cắt** → **dấu của hiệu hai phần tử liền kề**.
4. Quen với kỹ thuật "**cửa sổ ba phần tử**" (`nums[i-1], nums[i], nums[i+1]`) và việc chọn đúng miền chỉ số `1 .. n-2` để không bị `IndexError`.
5. Tránh các lỗi kinh điển: tính cả hai đầu mảng, dùng `>=` thay vì `>`, cộng đáy thay vì trừ, chỉ so với **một** hàng xóm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n`.

- Chỉ số `i` là **peak** nếu **đồng thời**: `0 < i < n - 1`, `nums[i] > nums[i - 1]` và `nums[i] > nums[i + 1]`.
- Chỉ số `i` là **valley** nếu **đồng thời**: `0 < i < n - 1`, `nums[i] < nums[i - 1]` và `nums[i] < nums[i + 1]`.

Trả về **hiệu** giữa **tổng giá trị các đỉnh** và **tổng giá trị các đáy** (tổng đỉnh − tổng đáy).

**Ví dụ 1**

- Input: `nums = [1,5,2,4,1,3,0]`
- Output: `9`
- Giải thích: các đỉnh là `5, 4, 3` (tại `i = 1, 3, 5`); các đáy là `2, 1` (tại `i = 2, 4`). `(5 + 4 + 3) - (2 + 1) = 9`.

**Ví dụ 2**

- Input: `nums = [-5,-6,-1,-2,5,3,2]`
- Output: `12`
- Giải thích: các đỉnh là `-1` (`i = 2`) và `5` (`i = 4`); các đáy là `-6` (`i = 1`) và `-2` (`i = 3`). `(-1 + 5) - (-6 + (-2)) = 12`.

**Ví dụ 3**

- Input: `nums = [6,5,2,5,5,3,6]`
- Output: `-5`
- Giải thích: không có đỉnh nào; các đáy là `2` (`i = 2`) và `3` (`i = 5`). `0 - (2 + 3) = -5`. Lưu ý `5, 5` ở giữa **bằng nhau** nên không phải đỉnh.

**Ràng buộc:** `3 <= n == nums.length <= 100`; `-100 <= nums[i] <= 100`.

In [1]:
from typing import List
import random

examples = [
    ([1, 5, 2, 4, 1, 3, 0], 9),
    ([-5, -6, -1, -2, 5, 3, 2], 12),
    ([6, 5, 2, 5, 5, 3, 6], -5),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[1, 5, 2, 4, 1, 3, 0] -> kỳ vọng 9
nums=[-5, -6, -1, -2, 5, 3, 2] -> kỳ vọng 12
nums=[6, 5, 2, 5, 5, 3, 6] -> kỳ vọng -5


## 3. Kiến thức nền

### 3.1. Danh sách (list), chỉ số và độ dài

`nums = [1, 5, 2]` là danh sách; `nums[0]` là phần tử đầu (chỉ số bắt đầu từ `0`), `nums[-1]` là phần tử cuối, `len(nums)` là độ dài. Chỉ số hợp lệ là `0 .. len(nums)-1`; đọc ngoài khoảng đó gây `IndexError`.

In [2]:
nums = [1, 5, 2, 4]
print(len(nums))          # 4
print(nums[0], nums[3])   # 1 4
print(nums[-1])           # 4 (phần tử cuối)
try:
    print(nums[4])        # ngoài khoảng
except IndexError as e:
    print("IndexError:", e)

4
1 4
4
IndexError: list index out of range


### 3.2. `range(a, b)` và vì sao đề chỉ xét `1 .. n-2`

`range(1, n - 1)` sinh ra `1, 2, ..., n-2` (số cuối `n-1` **không** có mặt). Đây đúng là các chỉ số "bên trong" mảng: những vị trí có **cả** hàng xóm trái (`i-1`) **lẫn** hàng xóm phải (`i+1`). Hai đầu mảng thiếu một hàng xóm nên đề loại ra.

In [3]:
n = 7
print(list(range(1, n - 1)))      # [1, 2, 3, 4, 5]  -> 5 vị trí bên trong
for i in range(1, n - 1):
    print(f"i={i}: hàng xóm trái = {i-1}, hàng xóm phải = {i+1}")

[1, 2, 3, 4, 5]
i=1: hàng xóm trái = 0, hàng xóm phải = 2
i=2: hàng xóm trái = 1, hàng xóm phải = 3
i=3: hàng xóm trái = 2, hàng xóm phải = 4
i=4: hàng xóm trái = 3, hàng xóm phải = 5
i=5: hàng xóm trái = 4, hàng xóm phải = 6


### 3.3. So sánh dây chuyền `a < b > c`

Python cho phép viết `a < b > c`, nghĩa là `(a < b) and (b > c)` (mỗi giá trị chỉ được đánh giá một lần). Với bài này, "`nums[i]` lớn hơn cả hai hàng xóm" viết gọn là `nums[i-1] < nums[i] > nums[i+1]`, còn "nhỏ hơn cả hai hàng xóm" là `nums[i-1] > nums[i] < nums[i+1]`. Hãy đọc nó như hình dạng: `<` rồi `>` giống một **ngọn đồi**, `>` rồi `<` giống một **cái hố**.

In [4]:
a, b, c = 1, 5, 2
print(a < b > c)          # True  (đỉnh: 1 < 5 > 2)
print(a > b < c)          # False
print(5 > 2 < 4)          # True  (đáy: 5 > 2 < 4)
print(3 < 5 > 5)          # False (5 > 5 sai: bằng nhau không phải "lớn hơn")

True
False
True
False


### 3.4. Bằng nhau thì sao? So sánh nghiêm ngặt

Đề dùng `>` và `<` (nghiêm ngặt). Với dãy `3, 5, 5, 2`, phần tử `5` đầu tiên không lớn hơn hàng xóm phải (`5 > 5` sai) nên **không** phải đỉnh; phần tử `5` thứ hai cũng không lớn hơn hàng xóm trái. Một "cao nguyên" (plateau) vì vậy không tạo ra đỉnh nào. Đây chính là điểm ví dụ 3 muốn kiểm tra.

In [5]:
plateau = [3, 5, 5, 2]
for i in range(1, len(plateau) - 1):
    print(i, plateau[i], "là đỉnh?", plateau[i - 1] < plateau[i] > plateau[i + 1])

1 5 là đỉnh? False
2 5 là đỉnh? False


### 3.5. `sum`, cộng dồn và `zip` với lát cắt

- `sum([4, 5, 3])` là `12`; `sum([])` là `0` (tổng rỗng bằng `0`, hữu ích khi không có đỉnh).
- `nums[1:]` là lát cắt bỏ phần tử đầu; `nums[2:]` bỏ hai phần tử đầu.
- `zip(a, b, c)` ghép phần tử cùng vị trí thành bộ ba và **dừng ở dãy ngắn nhất**. `zip(nums, nums[1:], nums[2:])` cho ra mọi bộ ba **liên tiếp** `(nums[i-1], nums[i], nums[i+1])`.

In [6]:
nums = [1, 5, 2, 4]
print(sum([4, 5, 3]), sum([]))
print(nums[1:], nums[2:])
print(list(zip(nums, nums[1:], nums[2:])))   # [(1, 5, 2), (5, 2, 4)]

12 0
[5, 2, 4] [2, 4]
[(1, 5, 2), (5, 2, 4)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy vẽ mảng `[1, 5, 2, 4, 1, 3, 0]` như một dãy núi: độ cao `1, 5, 2, 4, 1, 3, 0`.

| `i` | trái | giữa | phải | giữa so với hai bên | loại |
|---|---|---|---|---|---|
| 1 | 1 | 5 | 2 | `5 > 1` và `5 > 2` | peak (+5) |
| 2 | 5 | 2 | 4 | `2 < 5` và `2 < 4` | valley (−2) |
| 3 | 2 | 4 | 1 | `4 > 2` và `4 > 1` | peak (+4) |
| 4 | 4 | 1 | 3 | `1 < 4` và `1 < 3` | valley (−1) |
| 5 | 1 | 3 | 0 | `3 > 1` và `3 > 0` | peak (+3) |

Cộng/trừ: `5 - 2 + 4 - 1 + 3 = 9`. Cách nghĩ: **đi dọc mảng, nhìn từng bộ ba liên tiếp**; nếu phần tử giữa "nhô lên" thì cộng, nếu "lõm xuống" thì trừ, còn lại bỏ qua. Mỗi vị trí chỉ phụ thuộc vào **hai hàng xóm của nó**, nên không cần nhìn xa hơn.

## 5. Cách 1 — Brute Force / baseline: tách danh sách đỉnh và đáy

**Ý tưởng.** Làm đúng theo đề, từng bước một:

1. Viết hai hàm nhỏ `is_peak(nums, i)` và `is_valley(nums, i)` kiểm tra **đủ ba điều kiện** như đề.
2. Duyệt **mọi** chỉ số `i` từ `0` tới `n-1` (kể cả hai đầu, để điều kiện `0 < i < n-1` thực sự được kiểm tra), thu các giá trị đỉnh vào `peaks`, giá trị đáy vào `valleys`.
3. Trả `sum(peaks) - sum(valleys)`.

In [7]:
def is_peak(nums, i):
    return 0 < i < len(nums) - 1 and nums[i] > nums[i - 1] and nums[i] > nums[i + 1]


def is_valley(nums, i):
    return 0 < i < len(nums) - 1 and nums[i] < nums[i - 1] and nums[i] < nums[i + 1]


def diff_two_lists(nums):
    peaks, valleys = [], []
    for i in range(len(nums)):
        if is_peak(nums, i):
            peaks.append(nums[i])
        elif is_valley(nums, i):
            valleys.append(nums[i])
    return sum(peaks) - sum(valleys)


for nums, expected in examples:
    print(nums, "->", diff_two_lists(nums), "(kỳ vọng", expected, ")")

[1, 5, 2, 4, 1, 3, 0] -> 9 (kỳ vọng 9 )
[-5, -6, -1, -2, 5, 3, 2] -> 12 (kỳ vọng 12 )
[6, 5, 2, 5, 5, 3, 6] -> -5 (kỳ vọng -5 )


In [8]:
# Quan sát hai danh sách trung gian của ví dụ 1
nums = [1, 5, 2, 4, 1, 3, 0]
peaks = [nums[i] for i in range(len(nums)) if is_peak(nums, i)]
valleys = [nums[i] for i in range(len(nums)) if is_valley(nums, i)]
print("peaks  =", peaks, "tổng", sum(peaks))
print("valleys=", valleys, "tổng", sum(valleys))
print("hiệu   =", sum(peaks) - sum(valleys))

peaks  = [5, 4, 3] tổng 12
valleys= [2, 1] tổng 3
hiệu   = 9


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Duyệt `n` chỉ số, mỗi chỉ số kiểm tra `O(1)` so sánh: **Time `O(n)`**. Hai danh sách `peaks`, `valleys` có thể chứa tới `O(n)` phần tử: **Space `O(n)`**.

**Nút thắt.** Với `n <= 100` thì cách này hoàn toàn đủ nhanh; điểm yếu duy nhất là **dùng thêm bộ nhớ** cho hai danh sách mà ta chỉ cần **tổng** của chúng. Ta sẽ cộng/trừ ngay khi gặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1:** kết quả là `Σ(giá trị đỉnh) − Σ(giá trị đáy)`, tức mỗi **đỉnh đóng góp `+nums[i]`**, mỗi **đáy đóng góp `−nums[i]`**, các vị trí khác đóng góp `0`. Phép cộng có tính giao hoán và kết hợp nên ta **không cần lưu danh sách**, chỉ cần một biến `ans` cập nhật ngay khi gặp.

**Quan sát 2:** một vị trí không thể vừa là đỉnh vừa là đáy (`nums[i] > nums[i-1]` mâu thuẫn với `nums[i] < nums[i-1]`), nên dùng `if ... elif ...` là đủ.

**Quan sát 3:** chỉ cần duyệt `i` trong `1 .. n-2`, vì hai đầu mảng không bao giờ là đỉnh/đáy. Như vậy ta **không cần** viết điều kiện `0 < i < n-1` trong thân vòng lặp.

Kết quả: **một lượt duyệt, `O(n)` thời gian, `O(1)` bộ nhớ**.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt duyệt, cộng/trừ trực tiếp

**Giải thích.** `ans = 0`. Với `i` từ `1` tới `n-2`: nếu `nums[i-1] < nums[i] > nums[i+1]` thì `ans += nums[i]`; nếu `nums[i-1] > nums[i] < nums[i+1]` thì `ans -= nums[i]`.

In [9]:
def diff_single_pass(nums):
    ans = 0
    for i in range(1, len(nums) - 1):
        if nums[i - 1] < nums[i] > nums[i + 1]:
            ans += nums[i]            # đỉnh: cộng
        elif nums[i - 1] > nums[i] < nums[i + 1]:
            ans -= nums[i]            # đáy: trừ
    return ans


for nums, expected in examples:
    print(nums, "->", diff_single_pass(nums), "(kỳ vọng", expected, ")")

[1, 5, 2, 4, 1, 3, 0] -> 9 (kỳ vọng 9 )
[-5, -6, -1, -2, 5, 3, 2] -> 12 (kỳ vọng 12 )
[6, 5, 2, 5, 5, 3, 6] -> -5 (kỳ vọng -5 )


In [10]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def peakValleyDifference(self, nums: List[int]) -> int:
        ans = 0
        for i in range(1, len(nums) - 1):
            if nums[i - 1] < nums[i] > nums[i + 1]:
                ans += nums[i]
            elif nums[i - 1] > nums[i] < nums[i + 1]:
                ans -= nums[i]
        return ans


print(Solution().peakValleyDifference([1, 5, 2, 4, 1, 3, 0]))   # 9

9


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** (chỉ một biến `ans`).

### Cách 3 — `zip` ba lát cắt (bộ ba liên tiếp)

**Giải thích.** `zip(nums, nums[1:], nums[2:])` cho ra các bộ ba `(trái, giữa, phải)`; ta không cần đụng tới chỉ số. Lát cắt tạo bản sao nên tốn thêm bộ nhớ, đổi lại code ngắn và khó sai chỉ số.

In [11]:
def diff_zip(nums):
    ans = 0
    for left, mid, right in zip(nums, nums[1:], nums[2:]):
        if left < mid > right:
            ans += mid
        elif left > mid < right:
            ans -= mid
    return ans


for nums, expected in examples:
    print(nums, "->", diff_zip(nums), "(kỳ vọng", expected, ")")

[1, 5, 2, 4, 1, 3, 0] -> 9 (kỳ vọng 9 )
[-5, -6, -1, -2, 5, 3, 2] -> 12 (kỳ vọng 12 )
[6, 5, 2, 5, 5, 3, 6] -> -5 (kỳ vọng -5 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (hai lát cắt là hai bản sao dài `n-1` và `n-2`; có thể dùng `itertools.islice` để về `O(1)`).

### Cách 4 — Dấu của hiệu hai phần tử liền kề

**Giải thích.** Gọi `d[i] = nums[i+1] - nums[i]` (hiệu liền kề, dài `n-1`). Vị trí `i` là **đỉnh** khi nó "đi lên rồi đi xuống": `d[i-1] > 0` và `d[i] < 0`; là **đáy** khi "đi xuống rồi đi lên": `d[i-1] < 0` và `d[i] > 0`. Tư duy này mô tả đúng **hình dạng đồ thị** và là nền tảng của nhiều bài "dãy zigzag / núi".

In [12]:
def diff_by_slopes(nums):
    d = [nums[k + 1] - nums[k] for k in range(len(nums) - 1)]   # độ dốc giữa hai ô liền kề
    ans = 0
    for i in range(1, len(nums) - 1):
        up_before, down_after = d[i - 1] > 0, d[i] < 0
        down_before, up_after = d[i - 1] < 0, d[i] > 0
        if up_before and down_after:
            ans += nums[i]
        elif down_before and up_after:
            ans -= nums[i]
    return ans


print([ [b - a for a, b in zip(x, x[1:])] for x, _ in examples ])
for nums, expected in examples:
    print(nums, "->", diff_by_slopes(nums), "(kỳ vọng", expected, ")")

[[4, -3, 2, -3, 2, -3], [-1, 5, -1, 7, -2, -1], [-1, -3, 3, 0, -2, 3]]
[1, 5, 2, 4, 1, 3, 0] -> 9 (kỳ vọng 9 )
[-5, -6, -1, -2, 5, 3, 2] -> 12 (kỳ vọng 12 )
[6, 5, 2, 5, 5, 3, 6] -> -5 (kỳ vọng -5 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (mảng độ dốc `d`). Về mặt thực tế không hơn Cách 2, nhưng giúp bạn nhìn bài toán dưới dạng "đổi chiều đi lên ↔ đi xuống".

> Cách 2 là cách nộp tốt nhất: ngắn, `O(1)` bộ nhớ, đúng bản chất bài.

## 8. TRACE từng bước

Ta in trạng thái `ans` qua từng vòng lặp của Cách 2 cho ví dụ 1 và ví dụ 3 (có cao nguyên `5, 5`).

In [13]:
def trace_single_pass(nums):
    print("nums =", nums)
    ans = 0
    for i in range(1, len(nums) - 1):
        l, m, r = nums[i - 1], nums[i], nums[i + 1]
        if l < m > r:
            kind, ans = "peak  (+)", ans + m
        elif l > m < r:
            kind, ans = "valley(-)", ans - m
        else:
            kind = "không phải"
        print(f"i={i}: ({l:>3}, {m:>3}, {r:>3}) -> {kind:<11} ans={ans}")
    print("kết quả:", ans)
    return ans


trace_single_pass([1, 5, 2, 4, 1, 3, 0])
print()
trace_single_pass([6, 5, 2, 5, 5, 3, 6])

nums = [1, 5, 2, 4, 1, 3, 0]
i=1: (  1,   5,   2) -> peak  (+)   ans=5
i=2: (  5,   2,   4) -> valley(-)   ans=3
i=3: (  2,   4,   1) -> peak  (+)   ans=7
i=4: (  4,   1,   3) -> valley(-)   ans=6
i=5: (  1,   3,   0) -> peak  (+)   ans=9
kết quả: 9

nums = [6, 5, 2, 5, 5, 3, 6]
i=1: (  6,   5,   2) -> không phải  ans=0
i=2: (  5,   2,   5) -> valley(-)   ans=-2
i=3: (  2,   5,   5) -> không phải  ans=-2
i=4: (  5,   5,   3) -> không phải  ans=-2
i=5: (  5,   3,   6) -> valley(-)   ans=-5
kết quả: -5


-5

Bảng tóm tắt (ví dụ 1):

| `i` | bộ ba `(trái, giữa, phải)` | loại | `ans` sau bước |
|---|---|---|---|
| 1 | `(1, 5, 2)` | peak | 5 |
| 2 | `(5, 2, 4)` | valley | 3 |
| 3 | `(2, 4, 1)` | peak | 7 |
| 4 | `(4, 1, 3)` | valley | 6 |
| 5 | `(1, 3, 0)` | peak | 9 |

Ở ví dụ 3, bộ ba `(2, 5, 5)` và `(5, 5, 3)` đều rơi vào nhánh "không phải" vì có phép so sánh `5` với `5` không nghiêm ngặt.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (một lượt duyệt). Đúng `class Solution`, đúng tên method `peakValleyDifference`. Trên LeetCode `List` đã được import sẵn; trong notebook ta đã `from typing import List` ở trên.

In [14]:
class Solution:
    def peakValleyDifference(self, nums: List[int]) -> int:
        ans = 0
        for i in range(1, len(nums) - 1):
            if nums[i - 1] < nums[i] > nums[i + 1]:
                ans += nums[i]            # đỉnh
            elif nums[i - 1] > nums[i] < nums[i + 1]:
                ans -= nums[i]            # đáy
        return ans


sol = Solution()
for nums, expected in examples:
    assert sol.peakValleyDifference(nums) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Định nghĩa khớp từng chữ:** điều kiện `nums[i-1] < nums[i] > nums[i+1]` chính là hai vế `nums[i] > nums[i-1]` và `nums[i] > nums[i+1]` của đề; điều kiện đáy tương tự. Miền `i ∈ [1, n-2]` chính là `0 < i < n-1`.
- **Không đếm trùng, không bỏ sót:** mỗi chỉ số `i` được xét đúng một lần; đỉnh và đáy loại trừ nhau nên `elif` không làm mất trường hợp nào.
- **Cộng/trừ đúng dấu:** tổng đỉnh − tổng đáy = `Σ_{peak} nums[i] + Σ_{valley} (−nums[i])`, và vòng lặp cộng/trừ đúng như thế vào `ans`.
- **Bằng nhau:** dùng so sánh nghiêm ngặt giống đề, nên cao nguyên không bị tính (ví dụ 3).
- **Kích thước:** `n >= 3` nên luôn có ít nhất một vị trí bên trong; kể cả khi không có đỉnh/đáy nào, `ans = 0` vẫn đúng ("tổng rỗng" bằng 0).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa `100`).

- **Cách 1 (hai danh sách):** Time `O(n)`; Space `O(n)` (worst case: đỉnh/đáy xen kẽ nên danh sách dài ~ `n/2` mỗi cái; average cùng bậc).
- **Cách 2 (một lượt):** Time `O(n)`, Space `O(1)` — **tối ưu về bậc**, vì mọi phần tử bên trong đều phải được nhìn ít nhất một lần.
- **Cách 3 (`zip` lát cắt):** Time `O(n)`, Space `O(n)` do bản sao lát cắt.
- **Cách 4 (độ dốc):** Time `O(n)`, Space `O(n)` do mảng `d`.

Không có khác biệt "average / worst" vì thuật toán không phụ thuộc dữ liệu: luôn đúng `n - 2` lần kiểm tra.

## 12. Bộ test

Ta viết một **oracle brute force độc lập** (`oracle`) dùng đúng chữ định nghĩa của đề (`all`/`any` trên danh sách hàng xóm), rồi đối chiếu mọi cách bằng `assert`.

In [15]:
def oracle(nums):
    """Brute force độc lập: duyệt mọi i, kiểm tra đúng định nghĩa bằng danh sách hàng xóm."""
    n = len(nums)
    peak_sum = 0
    valley_sum = 0
    for i in range(n):
        if not (0 < i < n - 1):
            continue
        neighbors = [nums[i - 1], nums[i + 1]]
        if all(nums[i] > x for x in neighbors):
            peak_sum += nums[i]
        if all(nums[i] < x for x in neighbors):
            valley_sum += nums[i]
    return peak_sum - valley_sum


impls = {
    "two_lists": diff_two_lists,
    "single_pass": diff_single_pass,
    "zip": diff_zip,
    "slopes": diff_by_slopes,
    "submit": sol.peakValleyDifference,
}

for nums, expected in examples:
    assert oracle(nums) == expected
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, nums, fn(nums), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [16]:
edge_cases = [
    ([1, 2, 3], 0, "tăng dần: không đỉnh/đáy"),
    ([3, 2, 1], 0, "giảm dần"),
    ([1, 3, 1], 3, "đúng một đỉnh, n = 3"),
    ([3, 1, 3], -1, "đúng một đáy, n = 3"),
    ([2, 2, 2], 0, "ba số bằng nhau"),
    ([1, 3, 3, 1], 0, "cao nguyên: không đỉnh"),
    ([5, 1, 5, 1, 5], 5 - (1 + 1), "đáy i=1,3 (giá trị 1,1), đỉnh i=2 (giá trị 5)"),
    ([-100, 100, -100], 100, "giá trị biên của đề: đỉnh 100 tại i=1; hai đầu không tính"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, oracle(nums), expected)
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, label, nums, fn(nums), expected)
    print("PASS", label)

PASS tăng dần: không đỉnh/đáy
PASS giảm dần
PASS đúng một đỉnh, n = 3
PASS đúng một đáy, n = 3
PASS ba số bằng nhau
PASS cao nguyên: không đỉnh
PASS đáy i=1,3 (giá trị 1,1), đỉnh i=2 (giá trị 5)
PASS giá trị biên của đề: đỉnh 100 tại i=1; hai đầu không tính


### 12.2. Tính chất và phản ví dụ

- **Bất biến khi đảo dấu:** đổi `x -> -x` biến đỉnh thành đáy và ngược lại, đồng thời đổi dấu giá trị, nên tổng đỉnh − tổng đáy **không đổi**: `f(-nums) = f(nums)` (thử với `[1,5,2]`: đỉnh `5` -> đáy `-5`, `0 - (-5) = 5`).
- **Đảo ngược mảng:** hàng xóm trái/phải đổi chỗ nhưng điều kiện đối xứng, nên `f(nums[::-1]) = f(nums)`.
- **Phản ví dụ** cho một cách làm sai thường gặp: dùng `>=` thay vì `>` (coi cao nguyên là đỉnh).

In [17]:
random.seed(88)
for _ in range(500):
    n = random.randint(3, 12)
    nums = [random.randint(-5, 5) for _ in range(n)]
    base = sol.peakValleyDifference(nums)
    assert sol.peakValleyDifference([-x for x in nums]) == base
    assert sol.peakValleyDifference(nums[::-1]) == base
print("PASS tính chất bất biến (đảo dấu, đảo ngược) trên 500 mảng ngẫu nhiên")


def sai_dau_bang(nums):
    ans = 0
    for i in range(1, len(nums) - 1):
        if nums[i - 1] <= nums[i] >= nums[i + 1]:      # SAI: dùng >=, <=
            ans += nums[i]
        elif nums[i - 1] >= nums[i] <= nums[i + 1]:
            ans -= nums[i]
    return ans


bad = [6, 5, 2, 5, 5, 3, 6]
print("phản ví dụ:", bad, "-> đúng", sol.peakValleyDifference(bad), "| bản dùng >=:", sai_dau_bang(bad))
assert sai_dau_bang(bad) != sol.peakValleyDifference(bad)

PASS tính chất bất biến (đảo dấu, đảo ngược) trên 500 mảng ngẫu nhiên
phản ví dụ: [6, 5, 2, 5, 5, 3, 6] -> đúng -5 | bản dùng >=: 5


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [18]:
random.seed(4088)
for trial in range(3000):
    n = random.randint(3, 100)
    lo = random.choice([-100, -3, 0])
    hi = random.choice([100, 3, 1])
    if lo > hi:
        lo, hi = hi, lo
    nums = [random.randint(lo, hi) for _ in range(n)]
    truth = oracle(nums)
    for key, fn in impls.items():
        assert fn(nums) == truth, (key, nums, fn(nums), truth)
print("PASS 3000 mảng ngẫu nhiên (n tới 100, nhiều giá trị trùng): mọi cách khớp oracle")

PASS 3000 mảng ngẫu nhiên (n tới 100, nhiều giá trị trùng): mọi cách khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt cả hai đầu mảng

Viết `for i in range(len(nums))` rồi đọc `nums[i+1]` / `nums[i-1]` sẽ gây `IndexError` ở `i = n-1`, hoặc (với `i = 0`) **đọc nhầm** `nums[-1]` (phần tử cuối) vì chỉ số âm trong Python vẫn hợp lệ — lỗi im lặng, nguy hiểm hơn.

In [19]:
def sai_hai_dau(nums):
    ans = 0
    for i in range(len(nums)):                       # SAI: gồm i = 0 và i = n-1
        left = nums[i - 1]                           # i = 0 -> nums[-1] (phần tử cuối!)
        right = nums[(i + 1) % len(nums)]            # (cố tình bọc vòng để không văng lỗi)
        if left < nums[i] > right:
            ans += nums[i]
        elif left > nums[i] < right:
            ans -= nums[i]
    return ans


t = [9, 1, 2, 3, 4]
print("đúng:", Solution().peakValleyDifference(t), "| sai (coi mảng như vòng tròn):", sai_hai_dau(t))

đúng: -1 | sai (coi mảng như vòng tròn): 8


**Vì sao sai:** đề nói rõ `0 < i < n-1`; hai đầu **không** được tính. Cách đúng là cho `i` chạy `1 .. n-2` bằng `range(1, len(nums) - 1)`.

### Lỗi 2: Dùng `>=` / `<=` (coi cao nguyên là đỉnh)

Xem hàm `sai_dau_bang` ở phần test: với `[6,5,2,5,5,3,6]` nó cộng `5` hoặc trừ `5` nhầm. Cách đúng: so sánh **nghiêm ngặt** `>` và `<`.

### Lỗi 3: Sai dấu của đáy

In [20]:
def sai_cong_ca_hai(nums):
    ans = 0
    for i in range(1, len(nums) - 1):
        if nums[i - 1] < nums[i] > nums[i + 1]:
            ans += nums[i]
        elif nums[i - 1] > nums[i] < nums[i + 1]:
            ans += nums[i]                           # SAI: phải TRỪ giá trị đáy
    return ans


print("đúng:", Solution().peakValleyDifference([1, 5, 2, 4, 1, 3, 0]), "| sai:", sai_cong_ca_hai([1, 5, 2, 4, 1, 3, 0]))

đúng: 9 | sai: 15


**Vì sao sai:** đề yêu cầu **tổng đỉnh trừ tổng đáy**, nên đáy phải mang dấu `-`. Cộng cả hai cho `5+2+4+1+3 = 15` thay vì `9`.

### Lỗi 4: Chỉ so với **một** hàng xóm
#### Code sai

In [21]:
def sai_mot_ben(nums):
    ans = 0
    for i in range(1, len(nums) - 1):
        if nums[i] > nums[i - 1]:                    # SAI: thiếu so với nums[i+1]
            ans += nums[i]
        elif nums[i] < nums[i - 1]:
            ans -= nums[i]
    return ans


print("đúng:", Solution().peakValleyDifference([1, 2, 3, 0]), "| sai:", sai_mot_ben([1, 2, 3, 0]))

đúng: 3 | sai: 5


**Vì sao sai:** trong `[1, 2, 3, 0]`, số `2` chỉ lớn hơn hàng xóm trái nhưng **nhỏ hơn** hàng xóm phải nên không phải đỉnh; bản sai vẫn cộng. Đỉnh/đáy luôn cần **cả hai** phía. **Code đúng:** `nums[i-1] < nums[i] > nums[i+1]`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "so sánh một phần tử với **hai hàng xóm**" → nghĩ ngay tới **cửa sổ ba phần tử** và vòng lặp `range(1, n-1)`.
- Thấy "đỉnh / đáy / zigzag / núi / local maximum / local minimum" → xét **dấu của hiệu liền kề** hoặc so sánh dây chuyền `a < b > c`.
- Chữ **"strictly"** hay dấu `>`/`<` (không có dấu `=`) → cẩn thận các giá trị bằng nhau; hãy luôn tự thử một ca có cao nguyên.
- Cần **tổng** chứ không cần **danh sách** → cộng dồn trong vòng lặp, tiết kiệm bộ nhớ (`O(1)`).
- Ràng buộc nhỏ (`n <= 100`) cho phép cả cách chậm hơn; nhưng hãy tập viết cách `O(n)`/`O(1)` ngay từ đầu.
- Luôn kiểm tra tay ở **ba cỡ nhỏ**: `n = 3` (chỉ một vị trí bên trong), mảng tăng dần, mảng toàn bằng nhau.

## 15. Biến thể của bài

1. **Đếm** số đỉnh và số đáy thay vì tổng (trả cặp `(peaks, valleys)`).
2. **Chỉ số** của đỉnh cao nhất / đáy thấp nhất.
3. Mảng **vòng tròn** (phần tử đầu và cuối là hàng xóm của nhau): lúc đó hai đầu cũng có thể là đỉnh/đáy.
4. Cho phép **cao nguyên** (đỉnh = đoạn bằng nhau lớn hơn hai bên) — như LeetCode 162 / 852 / 1671 có các biến thể tìm đỉnh.
5. Tổng quát: **bán kính `k`** — đỉnh nếu lớn hơn mọi phần tử trong khoảng `k` ô mỗi phía.

In [22]:
def count_peaks_valleys(nums):
    peaks = valleys = 0
    for i in range(1, len(nums) - 1):
        if nums[i - 1] < nums[i] > nums[i + 1]:
            peaks += 1
        elif nums[i - 1] > nums[i] < nums[i + 1]:
            valleys += 1
    return peaks, valleys


def peak_valley_difference_circular(nums):
    """Biến thể: mảng vòng tròn, mọi vị trí đều có hai hàng xóm."""
    n = len(nums)
    ans = 0
    for i in range(n):
        l, m, r = nums[i - 1], nums[i], nums[(i + 1) % n]
        if l < m > r:
            ans += m
        elif l > m < r:
            ans -= m
    return ans


print(count_peaks_valleys([1, 5, 2, 4, 1, 3, 0]))             # (3, 2)
print(peak_valley_difference_circular([9, 1, 2, 3, 4]))       # 9 là đỉnh vòng tròn (4 < 9 > 1), 1 là đáy (9 > 1 < 2), 4 không

(3, 2)
8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai danh sách | thu đỉnh, đáy vào list rồi `sum` trừ | `O(n)` | `O(n)` | khi cần chính danh sách đỉnh/đáy để dùng tiếp |
| 2. Một lượt duyệt | cộng/trừ trực tiếp bằng `a < b > c` | `O(n)` | `O(1)` | **cách nộp chuẩn** |
| 3. `zip` lát cắt | duyệt bộ ba `(trái, giữa, phải)` | `O(n)` | `O(n)` | code ngắn, ít sai chỉ số |
| 4. Dấu độ dốc | đổi chiều `+ → −` là đỉnh, `− → +` là đáy | `O(n)` | `O(n)` | khi bài nói về "đi lên / đi xuống", zigzag |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `1 < 3 > 2`; (b) `2 > 2 < 5`; (c) `list(zip([1,2,3,4],[2,3,4],[3,4]))`; (d) `Solution().peakValleyDifference([1,3,1])`.

**Bài 2 (tính bằng tay).** Tìm đỉnh, đáy và đáp án của `nums = [2, 9, 4, 4, 7, 1, 8]`.

**Bài 3 (điền TODO).** Hoàn thiện `diff_todo` bên dưới (Cách 2).

In [23]:
def diff_todo(nums):
    ans = 0
    # TODO 1: duyệt i từ 1 tới len(nums) - 2 (nhớ range chặn trên)
    # TODO 2: nếu nums[i] lớn hơn cả hai hàng xóm -> cộng vào ans
    # TODO 3: ngược lại nếu nhỏ hơn cả hai hàng xóm -> trừ khỏi ans
    # TODO 4: trả ans
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert diff_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `diff_slopes2(nums)` chỉ dùng **dấu** của hiệu liền kề (ví dụ dùng hàm `sign(x)` trả `-1/0/1`) mà **không** dùng so sánh dây chuyền.

**Bài 5 (biến thể / mini challenge).** Viết `best_peak_and_valley(nums)` trả về `(đỉnh lớn nhất, đáy nhỏ nhất)` theo giá trị; trả `None` cho phần không tồn tại. Thử với `[6,5,2,5,5,3,6]` (không có đỉnh) và `[1,5,2,4,1,3,0]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: dây chuyền `a < b > c` đọc thành "`a < b` và `b > c`"; `zip` dừng ở dãy ngắn nhất. Bài 2: vẽ độ cao rồi xét từng bộ ba. Bài 5: gom đỉnh vào một danh sách riêng, đáy vào danh sách khác.
- **Mức 2 (vừa):** Bài 1: kết quả là `True`, `False`, `[(1,2,3),(2,3,4)]`, `3`. Bài 2: `i=1` (giá trị 9) là đỉnh; `i=2` (giá trị 4): `4 < 9` nhưng `4 < 4` sai nên không phải đáy; `i=3` (giá trị 4) cũng không (`4 > 4` sai và `4 < 4` sai); `i=4` (giá trị 7) là đỉnh; `i=5` (giá trị 1) là đáy. Bài 3: `for i in range(1, len(nums) - 1)`. Bài 4: `sign(d[i-1]) == 1 and sign(d[i]) == -1`.
- **Mức 3 (sâu):** Bài 2: `ans = 9 + 7 - 1 = 15`. Bài 3: `if nums[i-1] < nums[i] > nums[i+1]: ans += nums[i]` / `elif nums[i-1] > nums[i] < nums[i+1]: ans -= nums[i]`. Bài 5: `max(peaks) if peaks else None`, `min(valleys) if valleys else None`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert (1 < 3 > 2) is True and (2 > 2 < 5) is False
assert list(zip([1, 2, 3, 4], [2, 3, 4], [3, 4])) == [(1, 2, 3), (2, 3, 4)]
assert Solution().peakValleyDifference([1, 3, 1]) == 3

# Bài 2
nums2 = [2, 9, 4, 4, 7, 1, 8]
assert Solution().peakValleyDifference(nums2) == 9 + 7 - 1 == 15 and oracle(nums2) == 15


# Bài 3
def diff_todo_ans(nums):
    ans = 0
    for i in range(1, len(nums) - 1):
        if nums[i - 1] < nums[i] > nums[i + 1]:
            ans += nums[i]
        elif nums[i - 1] > nums[i] < nums[i + 1]:
            ans -= nums[i]
    return ans


for nums, expected in examples:
    assert diff_todo_ans(nums) == expected


# Bài 4
def sign(x):
    return (x > 0) - (x < 0)


def diff_slopes2(nums):
    d = [sign(nums[k + 1] - nums[k]) for k in range(len(nums) - 1)]
    ans = 0
    for i in range(1, len(nums) - 1):
        if d[i - 1] == 1 and d[i] == -1:
            ans += nums[i]
        elif d[i - 1] == -1 and d[i] == 1:
            ans -= nums[i]
    return ans


for nums, expected in examples:
    assert diff_slopes2(nums) == expected


# Bài 5
def best_peak_and_valley(nums):
    peaks = [nums[i] for i in range(1, len(nums) - 1) if nums[i - 1] < nums[i] > nums[i + 1]]
    valleys = [nums[i] for i in range(1, len(nums) - 1) if nums[i - 1] > nums[i] < nums[i + 1]]
    return (max(peaks) if peaks else None, min(valleys) if valleys else None)


assert best_peak_and_valley([6, 5, 2, 5, 5, 3, 6]) == (None, 2)
assert best_peak_and_valley([1, 5, 2, 4, 1, 3, 0]) == (5, 1)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ ba phần tử, duyệt `1 .. n-2`)

```python
class Solution:
    def peakValleyDifference(self, nums: List[int]) -> int:
        ans = 0
        for i in range(1, len(nums) - 1):
            if nums[i - 1] < nums[i] > nums[i + 1]:
                ans += nums[i]
            elif nums[i - 1] > nums[i] < nums[i + 1]:
                ans -= nums[i]
        return ans
```

**Checklist**

- [ ] `i` chạy `1 .. n-2` (`range(1, len(nums) - 1)`), hai đầu mảng không được tính?
- [ ] So sánh **nghiêm ngặt** (`<`, `>`), cao nguyên không phải đỉnh/đáy?
- [ ] Đỉnh **cộng**, đáy **trừ** (đáp án = tổng đỉnh − tổng đáy)?
- [ ] So với **cả hai** hàng xóm, không chỉ một bên?
- [ ] Đã thử: mảng tăng/giảm dần (`0`), `n = 3`, cao nguyên, giá trị âm?
- [ ] Đã đối chiếu với brute force theo định nghĩa trên mảng ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def peakValleyDifference(self, nums: List[int]) -> int`?